# Movie Review Sentiment Analysis - walkthrough

This notebook walks through the project pipeline using the project's own code:

**reviews → NLP preprocessing → TF-IDF → Logistic Regression → positive/negative → average → movie rating → Good/Average/Bad**

Run the scripts first (see README): `ml.download_data`, `ml.prepare_data`, `ml.train`, `ml.evaluate`.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from ml.config import MODEL_PATH, RESULTS_DIR, REVIEWS_PATH
from ml.preprocess import clean_text
from ml.rating import summarize_reviews

## 1. Dataset

Stanford aclImdb: 50,000 IMDb reviews, labelled positive (7-10 stars) or negative (1-4 stars).
The train and test splits contain different movies.

In [ ]:
reviews = pd.read_csv(REVIEWS_PATH)
print(reviews.shape)
print(reviews.groupby(["split", "label"]).size().unstack())
print("Reviews per test movie:")
print(reviews[reviews.split == "test"].groupby("imdb_id").size().describe())

## 2. NLP preprocessing

Lowercase, remove HTML, expand negations (didn't → did not), remove punctuation/numbers,
tokenize, and remove stopwords but keep negation words like *not*.

In [ ]:
examples = [
    "I didn't like this movie. It was NOT good!<br /><br />2/10",
    "The acting was brilliant and the story was wonderful.",
]
for text in examples:
    print("Original:", text)
    print("Cleaned: ", clean_text(text))

## 3. The trained model: TF-IDF + Logistic Regression

`ml/train.py` trained this pipeline on the 25,000 training reviews (C chosen by 5-fold cross-validation).
Logistic Regression learns one weight per word/word pair: positive weights push a review towards
*positive*, negative weights towards *negative*.

In [ ]:
model = joblib.load(MODEL_PATH)
vocabulary = model.named_steps["tfidf"].get_feature_names_out()
weights = model.named_steps["model"].coef_[0]
print("Number of TF-IDF features:", len(vocabulary))

words = pd.Series(weights, index=vocabulary).sort_values()
print("Most negative words:", list(words.index[:15]))
print("Most positive words:", list(words.index[-15:][::-1]))

## 4. Evaluation on the test set (movies never seen in training)

In [ ]:
print((RESULTS_DIR / "model_metrics.txt").read_text())
plt.imshow(plt.imread(RESULTS_DIR / "confusion_matrix.png"))
plt.axis("off")
plt.show()

## 5. From review predictions to a movie rating

Each review gets P(positive). The movie rating is `1 + 9 × average P(positive)`;
Good if rating ≥ 7, Average if 5-6.9, Bad if below 5.

In [ ]:
test = reviews[reviews.split == "test"]
movie_id = test.imdb_id.value_counts().index[0]  # a movie with many reviews
movie_reviews = test[test.imdb_id == movie_id]

probabilities = model.predict_proba(movie_reviews["text"].apply(clean_text))[:, 1]
print("Movie:", movie_id, "-", len(movie_reviews), "reviews")
print(summarize_reviews(list(probabilities)))
print("Reviewers' real average stars:", round(movie_reviews["stars"].mean(), 1))